# Anima 일반 LoRA 학습 · Colab

이미지와 동명 `.txt` 캡션 → 모델 준비 → TOML → 캐시 → 짧은 시험 학습(선택) → TensorBoard(선택) → 본 학습 → Drive 보관.

- DiffSynth의 LoRA 대상 탐색·플로우 매칭 손실을 활용합니다. 일반 A/B LoRA를 PyTorch로 학습합니다.
- 기본값: 해상도 512, batch 4, rank 32, alpha 128, AdamW 2e-5, 2,400 업데이트.
- batch 안의 이미지를 순차 계산하여 gradient를 평균합니다. 캡션의 실제 토큰 길이를 유지합니다.
- 이미지는 중앙 정사각형 crop을 적용합니다. 원본 이미지와 캡션은 수정하지 않습니다.
- CUDA GPU 런타임을 선택하세요. 메모리가 부족하면 batch·해상도·rank를 조절하세요.
- 정기 체크포인트는 런타임에 저장됩니다. 런타임 종료 전에 마지막 보관 셀을 실행하세요.


In [ ]:
#@title 1. 저장소와 학습 의존성 준비
from pathlib import Path
import importlib.util, subprocess, sys, shutil

REPO_URL = "https://github.com/HisameOgasahara/tmp_diffsynth_practive.git" #@param {type:"string"}
REPO_REVISION = "main" #@param {type:"string"}
ROOT = Path("/content/tmp_diffsynth_practive_training")
if not Path("/content").is_dir():
    raise RuntimeError("Google Colab에서 실행하세요.")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", REPO_REVISION], check=True)
else:
    revision = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True).strip()
    print("기존 작업 폴더를 사용합니다. commit:", revision)
    print("다른 revision으로 실행하려면 새 런타임에서 시작하세요.")
# uv가 없는 Colab에서 실행 도구만 bootstrap합니다. 학습 의존성은 uv로 설치합니다.
if importlib.util.find_spec("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--system",
                "--python", sys.executable, "-r", str(ROOT / "requirements-training.txt")], check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Colab 설정에서 GPU 런타임을 선택하세요.")
from packaging.version import Version
if Version(torch.__version__.split("+")[0]) < Version("2.4"):
    raise RuntimeError("torch >= 2.4가 필요합니다.")
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name())
print("저장소:", ROOT)


In [ ]:
#@title 2. 데이터·캐시·출력 경로와 Drive 연결
MOUNT_DRIVE = True #@param {type:"boolean"}
DATASET_DIR = "/content/drive/MyDrive/anima_dataset" #@param {type:"string"}
CACHE_DIR = "/content/anima_training/cache" #@param {type:"string"}
OUTPUT_ROOT = "/content/anima_training/output" #@param {type:"string"}
RUN_NAME = "anima_lora" #@param {type:"string"}
MODEL_CACHE_DIR = "/content/anima_training/models" #@param {type:"string"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME에는 폴더 이름 하나를 입력하세요.")
OUTPUT_DIR = Path(OUTPUT_ROOT).expanduser().resolve() / RUN_NAME
if not Path(DATASET_DIR).is_dir():
    raise ValueError("DATASET_DIR에 이미지와 동명 TXT 캡션 폴더를 지정하세요.")
print("데이터:", DATASET_DIR)
print("출력:", OUTPUT_DIR)


In [ ]:
#@title 3. 모델·토크나이저 다운로드
from anima_core.runtime import download_weights, load_tokenizers

WEIGHTS = download_weights(cache_dir=MODEL_CACHE_DIR)
QWEN_TOKENIZER, T5_TOKENIZER = load_tokenizers()
del QWEN_TOKENIZER, T5_TOKENIZER
for name, path in WEIGHTS.items():
    print(name, ":", path)


In [ ]:
#@title 4. TOML 불러오기·이번 실행 설정 표시
import json
from anima_core.training.config import load_config, resolve_dtype, write_config

CONFIG_PATH = "" #@param {type:"string"}
PRECISION_OVERRIDE = "" #@param ["", "bf16", "fp16", "no"]
# CONFIG_PATH가 비어 있으면 저장소의 configs/anima_lora.toml을 사용합니다.
# T4에서 bf16 실행이 지원되지 않으면 fp16을 선택하세요.
CONFIG_PATH = Path(CONFIG_PATH) if CONFIG_PATH else ROOT / "configs/anima_lora.toml"
CONFIG = load_config(CONFIG_PATH)
if PRECISION_OVERRIDE:
    CONFIG["runtime"]["mixed_precision"] = PRECISION_OVERRIDE
resolve_dtype(CONFIG, "cuda")
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print("유효 batch:", CONFIG["dataset"]["batch_size"] * CONFIG["training"]["gradient_accumulation_steps"])
print("warmup 업데이트:", round(CONFIG["training"]["max_steps"] * CONFIG["training"]["warmup_ratio"]))
# 재개할 때는 step 폴더의 training.toml을 CONFIG_PATH로 지정하세요.


In [ ]:
#@title 5. 이미지 latent·텍스트 조건 캐시 준비
from anima_core.training.dataset import prepare_cache, CachedDataset

CACHE_PATH = prepare_cache(CONFIG, DATASET_DIR, CACHE_DIR, WEIGHTS, device="cuda")
print("캐시 이미지 수:", len(CachedDataset(CACHE_PATH)))
print("캐시:", CACHE_PATH)


In [ ]:
#@title 6. 선택: 짧은 시험 학습
from copy import deepcopy
from datetime import datetime
from anima_core.training.trainer import train

RUN_SMOKE = False #@param {type:"boolean"}
SMOKE_STEPS = 3 #@param {type:"integer"}
if RUN_SMOKE:
    if SMOKE_STEPS < 1:
        raise ValueError("SMOKE_STEPS는 1 이상이어야 합니다.")
    smoke_config = deepcopy(CONFIG)
    smoke_config["training"]["max_steps"] = SMOKE_STEPS
    smoke_config["checkpoint"]["save_steps"] = SMOKE_STEPS
    SMOKE_DIR = Path(OUTPUT_ROOT) / ("smoke_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    SMOKE_RESULT = train(smoke_config, CACHE_PATH, WEIGHTS["dit"], SMOKE_DIR, tensorboard=False)
    print("시험 학습 완료:", SMOKE_RESULT)
    print("본 학습은 원본 모델과 새 LoRA·optimizer로 시작합니다.")
else:
    print("시험 학습을 건너뜁니다.")


In [ ]:
#@title 7. 선택: TensorBoard 열기
ENABLE_TENSORBOARD = True #@param {type:"boolean"}
if ENABLE_TENSORBOARD:
    ipython = get_ipython()
    ipython.run_line_magic("load_ext", "tensorboard")
    ipython.run_line_magic("tensorboard", '--logdir "' + str(OUTPUT_DIR / "logs") + '"')
    print("train/loss, train/learning_rate, train/seconds를 기록합니다.")


In [ ]:
#@title 8. 본 학습 또는 학습 재개
from anima_core.training.trainer import train
RESUME_FROM = "" #@param {type:"string"}
# 새 학습: 비워 둡니다. 재개: complete.json이 있는 step-... 폴더를 지정합니다.
# 재개 설정·데이터·모델이 저장 당시와 일치해야 합니다.
RESULT = train(CONFIG, CACHE_PATH, WEIGHTS["dit"], OUTPUT_DIR,
               resume_from=RESUME_FROM or None, tensorboard=globals().get("ENABLE_TENSORBOARD", False))
print("학습 완료:", RESULT)


In [ ]:
#@title 9. 결과 확인·Drive에 보관
import shutil
from datetime import datetime

SAVE_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/anima_training_results" #@param {type:"string"}
checkpoints = sorted(path for path in OUTPUT_DIR.glob("step-*") if (path / "complete.json").is_file())
if not checkpoints:
    raise RuntimeError("완료된 체크포인트가 없습니다.")
latest = checkpoints[-1]
print("생성용 LoRA:", latest / "lora.safetensors")
print("재개 폴더:", latest)
print("재개 설정:", latest / "training.toml")
print("모든 정기 체크포인트:", [path.name for path in checkpoints])
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination_root = Path(DRIVE_OUTPUT_DIR).expanduser().resolve()
    drive_root = Path("/content/drive/MyDrive").resolve()
    if not destination_root.is_relative_to(drive_root):
        raise ValueError("DRIVE_OUTPUT_DIR는 MyDrive 안으로 지정하세요.")
    destination = destination_root / (RUN_NAME + "_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    partial = destination.with_name(destination.name + ".partial")
    partial.mkdir(parents=True, exist_ok=False)
    shutil.copytree(latest, partial / latest.name)
    for filename in ("training.toml", "lora_targets.json", "metrics.jsonl"):
        source = OUTPUT_DIR / filename
        if source.is_file():
            shutil.copy2(source, partial / filename)
    if (OUTPUT_DIR / "logs").is_dir():
        shutil.copytree(OUTPUT_DIR / "logs", partial / "logs")
    partial.rename(destination)
    print("마지막 체크포인트와 로그를 보관했습니다:", destination)
    print("재개 폴더:", destination / latest.name)


## 다음 실행

1. 원본 이미지·TXT 캡션과 모델을 준비합니다.
2. 보관한 step 폴더의 `training.toml`을 설정 셀에 지정합니다.
3. 동일한 전처리 설정으로 캐시를 준비합니다.
4. 본 학습 셀의 `RESUME_FROM`에 보관한 step 폴더를 지정합니다.

재개 상태에는 optimizer, scheduler, scaler, RNG, 데이터 순서가 포함됩니다. 동일한 설정·캐시를 사용해야 하며, 다른 GPU·PyTorch 버전에서 수치가 완전히 같음을 보장하지 않습니다. 캐시 식별에는 원본 파일 경로·크기·수정 시각과 모델 파일 정보가 포함됩니다.

학습 셀을 수동 중단했다면 마지막으로 완료된 정기 체크포인트에서 재개합니다. 진행 중인 업데이트는 저장되지 않습니다. 중단 후에도 결과 보관 셀은 실행할 수 있습니다.

참고: [DiffSynth-Studio](https://github.com/modelscope/DiffSynth-Studio), [학습 설정 참고 노트북](https://github.com/HisameOgasahara/irodori_test/blob/main/anima_lora_cli_colab.ipynb).
